# Governed Market Memory MCP — Five-Minute Classroom Laboratory

**Purpose.** Reconstruct the complete workflow transparently: configuration → provider → governance gates → predecessor memory → five reports → final narrative → cryptographic audit bundle → MCP exposure.

The default observations are **SIMULATED**. The notebook never describes them as live Yahoo Finance quotations.

## Learning objectives

By the end, students can explain why MCP supplies controlled capabilities but not a clock; why an orchestrator schedules cycles; how an Obsidian vault supplies cumulative memory; and why transparency must be strengthened into auditability and authorization.

In [1]:
# Colab setup: mount Drive only when running in Google Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print({'in_colab': IN_COLAB})

Mounted at /content/drive
{'in_colab': True}


In [2]:
# Choose the working directory. Change DRIVE_FOLDER to your mounted folder when needed.
from pathlib import Path
DRIVE_FOLDER = Path('/content/drive/MyDrive/THE ESSENTIAL WORKFLOWS/THE FINANCIAL REPORTER') if IN_COLAB else Path.cwd()
PACKAGE_ROOT = DRIVE_FOLDER / 'Governed_Market_Memory_MCP_Classroom'
VAULT = PACKAGE_ROOT / 'Market_Day_Obsidian_Vault'
PACKAGE_ROOT.mkdir(parents=True, exist_ok=True)
print({'package_root': str(PACKAGE_ROOT), 'vault': str(VAULT)})

{'package_root': '/content/drive/MyDrive/THE ESSENTIAL WORKFLOWS/THE FINANCIAL REPORTER/Governed_Market_Memory_MCP_Classroom', 'vault': '/content/drive/MyDrive/THE ESSENTIAL WORKFLOWS/THE FINANCIAL REPORTER/Governed_Market_Memory_MCP_Classroom/Market_Day_Obsidian_Vault'}


## Architecture

The provider observes. The governance engine decides admissibility. The vault persists memory. The orchestrator supplies the clock. The MCP server exposes bounded tools. A model may interpret accepted evidence but may not override failed gates.

In [3]:
# Install the package after copying/unzipping this classroom folder to Drive.
import os, sys, subprocess
if (PACKAGE_ROOT / 'pyproject.toml').exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PACKAGE_ROOT)], check=True)
else:
    print('Place the extracted package at', PACKAGE_ROOT)
    print('For a local run, execute this notebook from the package directory.')

Place the extracted package at /content/drive/MyDrive/THE ESSENTIAL WORKFLOWS/THE FINANCIAL REPORTER/Governed_Market_Memory_MCP_Classroom
For a local run, execute this notebook from the package directory.


In [4]:
# Import governed components.
from market_mcp.core import UNIVERSE, SimulatedProvider, GovernanceEngine, VaultStore
from market_mcp.orchestrator import MarketDayOrchestrator
print(UNIVERSE)

ModuleNotFoundError: No module named 'market_mcp'

## Source and limitation register

`SimulatedProvider` generates five deterministic pedagogical states. The optional `YFinanceProvider` is an unofficial Yahoo-facing adapter, may be delayed or rate-limited, and is not an OpenAI or official Yahoo connector.

In [ ]:
provider = SimulatedProvider()
metadata = {
    'provider': provider.name,
    'classification': 'SIMULATED',
    'official_yahoo_connector': False,
    'purpose': 'compressed classroom market day'
}
metadata

## Gate 1 — authorization failure is visible

We first prove that the write boundary rejects an incorrect token. A governed system must demonstrate refusal, not merely successful execution.

In [ ]:
engine = GovernanceEngine()
candidate = provider.snapshot(1)
rejected = engine.evaluate(candidate, 'WRONG-TOKEN', predecessor_count=0, cycle=1)
assert rejected['accepted'] is False
rejected

## Inspect the candidate observation

Each row carries symbol, human-readable name, level, change, UTC timestamp, classification and provider. No interpretation is stored inside the observation record.

In [ ]:
from dataclasses import asdict
import pandas as pd
pd.DataFrame([asdict(x) for x in candidate])

## Run cycle 1

The first report correctly records that no predecessor exists. It therefore establishes the initial state without fabricating memory.

In [ ]:
store = VaultStore(VAULT)
decision1 = engine.evaluate(candidate, 'CLASSROOM-AUTHORIZED', len(store.previous_reports()), 1)
report1 = store.save_cycle(1, candidate, decision1)
print(report1.read_text())

## Run cycles 2–5 with predecessor retrieval

Before each write, the gate requires exactly `cycle − 1` reports. Each Markdown report lists the Obsidian links it consulted.

In [ ]:
cycle_evidence = []
for cycle in range(2, 6):
    observations = provider.snapshot(cycle)
    predecessors = store.previous_reports()
    decision = engine.evaluate(observations, 'CLASSROOM-AUTHORIZED', len(predecessors), cycle)
    path = store.save_cycle(cycle, observations, decision)
    cycle_evidence.append({'cycle': cycle, 'predecessors': [p.stem for p in predecessors], 'accepted': decision['accepted'], 'report': path.name})
pd.DataFrame(cycle_evidence)

## Path-dependent synthesis

The final narrative uses the complete accepted chain. Its central claim is not merely the closing direction, but the intraday transition from broad stress to selective risk appetite and geographic/style divergence.

In [ ]:
final_path = store.finalize()
print(final_path.read_text())

## Audit manifest and cryptographic integrity

The manifest records every artifact path, byte count and SHA-256 digest. Recalculation turns a vague promise of traceability into an executable test.

In [ ]:
manifest_path = store.manifest()
verification = store.verify()
assert verification['valid']
verification

## Validate semantics and lineage

These tests fail visibly if the first report invents a predecessor, the fifth report omits the fourth, any classification disappears, or fewer than five accepted reports exist.

In [ ]:
reports = store.previous_reports()
assert len(reports) == 5
assert 'None — first observation' in reports[0].read_text()
assert '[[Minute_04]]' in reports[4].read_text()
assert all('classification: SIMULATED' in p.read_text() for p in reports)
assert (VAULT / '04_Audit/events.jsonl').exists()
print('All semantic, lineage, classification and audit checks passed.')

## MCP tool surface

`market_mcp.server` exposes read tools for universe, snapshots, status, metadata, validation and prior reports; controlled write tools create reports, finalize the narrative, generate the manifest and verify integrity.

Start locally with `python -m market_mcp.server`. A remote deployment needs HTTPS and appropriate authentication. MCP waits for calls; it does not schedule itself.

In [ ]:
# Display the governed tool names from source without starting a blocking server.
tool_names = [
 'get_market_universe', 'get_market_snapshot', 'get_market_status',
 'get_provider_metadata', 'validate_snapshot', 'read_previous_reports',
 'save_snapshot', 'save_report', 'finalize_market_day',
 'write_audit_manifest', 'verify_vault_integrity'
]
tool_names

## Optional five-real-minute execution

For a live classroom performance, start with a fresh vault and run the orchestrator with `interval_seconds=60`. Do not rerun against the completed vault because the memory gate will correctly reject duplicate sequencing.

In [ ]:
# Uncomment only for a fresh five-minute classroom performance.
# fresh_vault = PACKAGE_ROOT / 'Five_Real_Minute_Run'
# result = MarketDayOrchestrator(fresh_vault).run(cycles=5, interval_seconds=60)
# result['verification']

## Export the audit bundle

The ZIP is a portable evidence package. The working vault remains ordinary Markdown/CSV/JSON and can be opened directly in Obsidian.

In [ ]:
import shutil
zip_path = shutil.make_archive(str(PACKAGE_ROOT / 'Market_Day_Audit_Bundle'), 'zip', VAULT)
print(zip_path)

## Production extension

Replace the simulated provider with a licensed adapter; replace Colab with a persistent scheduler; implement market calendars, explicit freshness thresholds, OAuth 2.1, secrets management, durable telemetry, retries, retention and human review. Keep reporting isolated from trading authority.

Official OpenAI references:

- https://developers.openai.com/apps-sdk/build/mcp-server
- https://developers.openai.com/apps-sdk/concepts/mcp-server
- https://developers.openai.com/api/docs/guides/tools-connectors-mcp
- https://developers.openai.com/apps-sdk/build/auth